## 1. Extract EPA Monitoring Stations

Extract distinct EPA monitoring stations from the Silver layer, including station IDs, county codes, and geographic coordinates.

In [ ]:
df_silver = spark.table("silver_epa_no2_hourly")

df_stations = df_silver.select(
    "station_id",
    "county_code",
    "latitude",
    "longitude"
).distinct()

print("Station records:", df_stations.count())
df_stations.show(truncate=False)

## 2. Build Hourly EPA NO2 Fact

Aggregate Silver EPA NO2 measurements by station and UTC hour.

Calculate the average NO2 concentration and measurement count for hourly air quality analysis.

In [ ]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

df_no2_hourly = (
    spark.table("silver_epa_no2_hourly")
    .withColumn(
        "measurement_hour_utc",
        F.date_trunc("hour", F.col("measurement_utc"))
    )
    .groupBy("station_id", "measurement_hour_utc")
    .agg(
        F.avg("no2_ppb").alias("avg_no2_ppb"),
        F.count("*").alias("measurement_count")
    )
)

print("Gold hourly rows:", df_no2_hourly.count())

df_no2_hourly.show(5, truncate=False)

## 3. Convert UTC to New York Local Time

Convert hourly EPA NO2 timestamps from UTC to America/New_York time.

Extract local dates and hours to support time-based comparisons with NYC Taxi data.

In [ ]:
df_no2_local = (
    df_no2_hourly
    .withColumn(
        "measurement_hour_ny",
        F.from_utc_timestamp(
            "measurement_hour_utc",
            "America/New_York"
        )
    )
    .withColumn(
        "measurement_date_ny",
        F.to_date("measurement_hour_ny")
    )
    .withColumn(
        "hour_ny",
        F.hour("measurement_hour_ny")
    )
)

df_no2_local.select(
    "station_id",
    "measurement_hour_utc",
    "measurement_hour_ny",
    "hour_ny",
    "avg_no2_ppb"
).show(5, truncate=False)

## 4. Save Gold EPA NO2 Hourly Fact

Prepare the final hourly NO2 Fact table with UTC and New York timestamps, local date, hour, average concentration, and measurement count.

Save the data as a Gold Delta table and verify the persisted record count.

In [ ]:
df_no2_gold = df_no2_local.select(
    "station_id",
    "measurement_hour_utc",
    "measurement_hour_ny",
    "measurement_date_ny",
    "hour_ny",
    "avg_no2_ppb",
    "measurement_count"
).filter(
    F.col("measurement_date_ny").between(
        "2025-06-01", "2025-07-31"
    )
)

(
    df_no2_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_fact_epa_no2_hourly")
)

print(
    "Saved rows:",
    spark.table("gold_fact_epa_no2_hourly").count()
)

## 5. Prepare Spatial Mapping

Install Shapely and load NYC Taxi Zone boundaries from GeoJSON.

Prepare EPA station coordinates and geographic polygons for spatial matching.

In [ ]:
%pip install shapely

In [ ]:
import requests
from shapely.geometry import Point, shape


geojson_url = (
    "https://data.cityofnewyork.us/api/v3/views/"
    "8meu-9t5y/query.geojson?accessType=DOWNLOAD"
)

response = requests.get(geojson_url, timeout=90)
response.raise_for_status()

features = response.json()["features"]

print("EPA stations:", df_stations.count())
print("Taxi zones:", len(features))

## 6. Match EPA Stations to NYC Taxi Zones

Use Shapely point-in-polygon matching to assign each EPA monitoring station to an NYC Taxi Zone based on geographic coordinates.

Extract the matching LocationID, zone name, and borough for further analysis.

In [ ]:
station_zone_matches = []

for station in df_stations.collect():
    point = Point(
        station["longitude"],
        station["latitude"]
    )

    for feature in features:
        polygon = shape(feature["geometry"])

        if polygon.covers(point):
            station_zone_matches.append({
                "station_id": station["station_id"],
                "LocationID": int(feature["properties"]["locationid"]),
                "Zone": feature["properties"]["zone"],
                "Borough": feature["properties"]["borough"]
            })

for match in station_zone_matches:
    print(match)

print("Matched stations:", len(station_zone_matches))

## 7. Validate Station-to-Zone Mapping

Convert the spatial mapping results into a Spark DataFrame.

Validate the mapped LocationIDs against the Gold Zone Dimension and identify any unmatched zones.

In [ ]:
df_station_zones = spark.createDataFrame(
    [
        (m["station_id"], m["LocationID"])
        for m in station_zone_matches
    ],
    ["station_id", "LocationID"]
)

df_verified = df_station_zones.join(
    spark.table("gold_dim_zone"),
    on="LocationID",
    how="left"
)

df_verified.orderBy("station_id").show(truncate=False)

print(
    "Unmatched zones:",
    df_verified.filter(F.col("Zone").isNull()).count()
)

## 8. Update and Verify EPA Station Dimension

Enrich the EPA Station Dimension with NYC Taxi LocationIDs using a Delta MERGE operation.

Validate the station count and ensure all monitoring stations have valid zone mappings.

In [ ]:
df_dim_epa_station = (
    df_stations
    .join(df_station_zones, on="station_id", how="left")
    .withColumn("LocationID", F.col("LocationID").cast("int"))
)

print(
    "Missing LocationID:",
    df_dim_epa_station.filter(F.col("LocationID").isNull()).count()
)

(
    df_dim_epa_station.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_dim_epa_station")
)

df_updated = spark.table("gold_dim_epa_station")

print("Saved stations:", df_updated.count())
df_updated.orderBy("station_id").show(truncate=False)